In [ ]:
# ------------------ INSTALL AND IMPORT ------------------
!pip install -q h2o shap openpyxl scikit-learn pandas matplotlib seaborn

import h2o
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from functools import reduce
from h2o.automl import H2OAutoML
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, jaccard_score, cohen_kappa_score,
    f1_score, recall_score, precision_score, matthews_corrcoef, roc_auc_score
)

# ------------------ USER SETTINGS ------------------
file_path = "dataset"
true_label_col = "outcome variable" #poverty cluster label/AFA label

n_latent = 2
n_clusters = 2
num_runs = 20
test_size = 0.3
max_models = 10
seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61]
excel_out = "experimental_summary.xlsx"
baseline_families = ["GLM", "GBM", "DRF", "XGBoost", "deep learning"]

raw_data = pd.read_excel(file_path).dropna()
feature_cols = [c for c in raw_data.columns if c != true_label_col]

# ------------------ HELPER FUNCTIONS ------------------
def get_prob_matrix(pred_df):
    prob_cols = [c for c in pred_df.columns if c.startswith('p')]
    classes = [c.replace('p', '') for c in prob_cols]
    P = pred_df[prob_cols].to_numpy()
    return P, classes

def multiclass_auc(y_true, pred_df):
    P, classes = get_prob_matrix(pred_df)
    y_true_bin = pd.get_dummies(pd.Categorical(y_true, categories=classes))
    y_true_bin = y_true_bin[classes]
    return roc_auc_score(y_true_bin.values, P, average='macro', multi_class='ovo')

def per_class_metrics(y_true, y_pred):
    labels = np.unique(y_true)
    recalls = recall_score(y_true, y_pred, average=None, labels=labels)
    precisions = precision_score(y_true, y_pred, average=None, labels=labels)
    f1s = f1_score(y_true, y_pred, average=None, labels=labels)
    rows = []
    for i, lab in enumerate(labels):
        rows.append({
            "label": lab,
            "recall": recalls[i],
            "precision": precisions[i],
            "f1": f1s[i]
        })
    return pd.DataFrame(rows)

def safe_varimp_df(model):
    try:
        return model.varimp(use_pandas=True)
    except Exception:
        return pd.DataFrame(columns=["variable", "relative_importance", "scaled_importance", "percentage"])

def model_params_to_df(model):
    rows = []
    for k, v in model.params.items():
        rows.append({"param": k, "actual": v.get("actual", None), "default": v.get("default", None), "type": v.get("type", None)})
    return pd.DataFrame(rows)

def model_family_from_id(model_id):
    if model_id.startswith("StackedEnsemble"):
        return "StackedEnsemble"
    return model_id.split('_', 1)[0]

def compute_metrics_all(y_true, y_pred, pred_df):
    precision = precision_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')
    jaccard = jaccard_score(y_true, y_pred, average='macro')
    kappa = cohen_kappa_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    gmean = np.sqrt(recall * precision)
    try:
        auc_macro = multiclass_auc(y_true, pred_df)
    except Exception:
        auc_macro = np.nan
    return {"precision": precision, "recall": recall, "f1": f1, "jaccard": jaccard,
            "kappa": kappa, "mcc": mcc, "gmean": gmean, "auc_macro": auc_macro}

# ------------------ STORAGE ------------------
all_runs_metrics = []
confusion_matrices_tbl = []
classwise_tbl = []
feature_tables = []
params_tables = []
family_auc_tracker = {}

baseline_runs_metrics = []
baseline_confusion_tbl = []
baseline_classwise_tbl = []
baseline_feature_tables = []
baseline_params_tables = []
baseline_family_auc_tracker = {}

# ------------------ START H2O ------------------
h2o.init(nthreads=-1, max_mem_size="4G")

# ------------------ MAIN LOOP ------------------
for run_idx, seed in enumerate(seeds):
    print(f"\n🎯 Experimental Run {run_idx+1} (Seed={seed})")

    train_df, test_df = train_test_split(
        raw_data,
        test_size=test_size,
        random_state=seed,
        stratify=raw_data[true_label_col]
    )

    scaler = StandardScaler()
    fa = FactorAnalysis(n_components=n_latent, random_state=seed)
    X_train_latent = fa.fit_transform(scaler.fit_transform(train_df[feature_cols]))
    X_test_latent = fa.transform(scaler.transform(test_df[feature_cols]))

    kmeans = KMeans(n_clusters=n_clusters, random_state=seed)
    train_df["cluster_label"] = kmeans.fit_predict(X_train_latent).astype(str)
    test_df["cluster_label"] = kmeans.predict(X_test_latent).astype(str)

    train_h2o = h2o.H2OFrame(train_df)
    test_h2o = h2o.H2OFrame(test_df)

    train_h2o[true_label_col] = train_h2o[true_label_col].asfactor()
    test_h2o[true_label_col] = test_h2o[true_label_col].asfactor()
    train_h2o["cluster_label"] = train_h2o["cluster_label"].asfactor()
    test_h2o["cluster_label"] = test_h2o["cluster_label"].asfactor()

    targets = [true_label_col, "cluster_label"]

    # ---------- MAIN AutoML ----------
    for target in targets:
        print(f"\n🔧 Training AutoML for: {target}")
        aml = H2OAutoML(
            max_models=max_models,
            seed=seed,
            nfolds=5,
            include_algos=["GLM", "GBM", "DRF", "XGBoost", "DeepLearning"],
            sort_metric="AUC"
        )
        aml.train(x=feature_cols, y=target, training_frame=train_h2o)

        leaderboard = aml.leaderboard.as_data_frame()
        print(leaderboard[["model_id", "auc", "logloss", "mean_per_class_error"]])

        family_auc_tracker.setdefault(target, {})
        for _, row in leaderboard.iterrows():
            family = model_family_from_id(row["model_id"])
            auc_val = row.get("auc", np.nan)
            if np.isnan(auc_val):
                continue
            family_auc_tracker[target].setdefault(family, [np.nan]*num_runs)
            family_auc_tracker[target][family][run_idx] = auc_val

        pred_df = aml.leader.predict(test_h2o).as_data_frame()
        y_true = test_h2o[target].as_data_frame().values.flatten().astype(str)
        y_pred = pred_df["predict"].values.flatten().astype(str)
        metrics = compute_metrics_all(y_true, y_pred, pred_df)

        all_runs_metrics.append({
            "run": run_idx + 1,
            "target": target,
            "model_id": aml.leader.model_id,
            "leader_algo": aml.leader.algo,
            **metrics
        })

        confusion_matrices_tbl.append({
            "run": run_idx + 1,
            "target": target,
            "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
        })
        classwise_tbl.append(per_class_metrics(y_true, y_pred).assign(run=run_idx + 1, target=target))
        feature_tables.append(safe_varimp_df(aml.leader).assign(run=run_idx + 1, target=target))
        params_tables.append(model_params_to_df(aml.leader).assign(run=run_idx + 1, target=target))

    # ---------- BASELINE AutoML ----------
    for fam in baseline_families:
        print(f"\n🧪 Baseline AutoML ({fam}) for: {true_label_col}")
        aml_base = H2OAutoML(
            max_models=max_models,
            seed=seed,
            nfolds=5,
            include_algos=[fam],
            sort_metric="AUC"
        )
        aml_base.train(x=feature_cols, y=true_label_col, training_frame=train_h2o)

        pred_df_base = aml_base.leader.predict(test_h2o).as_data_frame()
      ########y_true = test_h2o[true_label_col].as_data_frame().values.flatten().astype(str)
        y_true = test_h2o[true_label_col].as_data_frame().values.flatten().astype(str)
        y_pred = pred_df_base["predict"].values.flatten().astype(str)
        metrics_base = compute_metrics_all(y_true, y_pred, pred_df_base)

        baseline_runs_metrics.append({
            "run": run_idx + 1,
            "target": true_label_col,
            "model_id": aml_base.leader.model_id,
            "leader_algo": aml_base.leader.algo,
            "baseline_family": fam,
            "auc_macro": metrics_base["auc_macro"],
            **metrics_base
        })

        baseline_confusion_tbl.append({
            "run": run_idx + 1,
            "target": true_label_col,
            "baseline_family": fam,
            "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
        })

        baseline_classwise_tbl.append(
            per_class_metrics(y_true, y_pred).assign(run=run_idx + 1, target=true_label_col, baseline_family=fam)
        )

        baseline_feature_tables.append(
            safe_varimp_df(aml_base.leader).assign(run=run_idx + 1, target=true_label_col, baseline_family=fam)
        )

        baseline_params_tables.append(
            model_params_to_df(aml_base.leader).assign(run=run_idx + 1, target=true_label_col, baseline_family=fam)
        )

        baseline_family_auc_tracker.setdefault(true_label_col, {})
        baseline_family_auc_tracker[true_label_col].setdefault(fam, [np.nan]*num_runs)
        baseline_family_auc_tracker[true_label_col][fam][run_idx] = metrics_base["auc_macro"]
# ------------------ EXPORT TO EXCEL ------------------
summary_df = pd.DataFrame(all_runs_metrics)
baseline_summary_df = pd.DataFrame(baseline_runs_metrics)

confusion_df = pd.DataFrame(confusion_matrices_tbl)
classwise_df = pd.concat(classwise_tbl, ignore_index=True) if classwise_tbl else pd.DataFrame()
feature_df = pd.concat(feature_tables, ignore_index=True) if feature_tables else pd.DataFrame()
params_df = pd.concat(params_tables, ignore_index=True) if params_tables else pd.DataFrame()

baseline_confusion_df = pd.DataFrame(baseline_confusion_tbl)
baseline_classwise_df = pd.concat(baseline_classwise_tbl, ignore_index=True) if baseline_classwise_tbl else pd.DataFrame()
baseline_feature_df = pd.concat(baseline_feature_tables, ignore_index=True) if baseline_feature_tables else pd.DataFrame()
baseline_params_df = pd.concat(baseline_params_tables, ignore_index=True) if baseline_params_tables else pd.DataFrame()

baseline_auc_df = pd.DataFrame([
    {
        "run": m["run"],
        "baseline_family": m["baseline_family"],
        "auc_macro": m["auc_macro"]
    }
    for m in baseline_runs_metrics
])

with pd.ExcelWriter(excel_out, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name="summary", index=False)
    baseline_summary_df.to_excel(writer, sheet_name="baseline_summary", index=False)
    baseline_auc_df.to_excel(writer, sheet_name="baseline_auc", index=False)
    confusion_df.to_excel(writer, sheet_name="confusion_matrices", index=False)
    classwise_df.to_excel(writer, sheet_name="classwise_metrics", index=False)
    feature_df.to_excel(writer, sheet_name="feature_importance", index=False)
    params_df.to_excel(writer, sheet_name="model_parameters", index=False)
    baseline_confusion_df.to_excel(writer, sheet_name="baseline_confusion", index=False)
    baseline_classwise_df.to_excel(writer, sheet_name="baseline_classwise", index=False)
    baseline_feature_df.to_excel(writer, sheet_name="baseline_features", index=False)
    baseline_params_df.to_excel(writer, sheet_name="baseline_params", index=False)

print(f"\n✅ All outputs saved to: {excel_out}")
# ------------------ LEARNING CURVES ------------------
for target, fam_dict in family_auc_tracker.items():
    plt.figure(figsize=(10, 6))
    for fam, auc_list in fam_dict.items():
        plt.plot(range(1, num_runs + 1), auc_list, marker='o', label=fam)
    plt.title(f"Learning Curve (AUC across runs) — {target}")
    plt.xlabel("Run")
    plt.ylabel("AUC")
    plt.xticks(range(1, num_runs + 1))
    plt.grid(True, alpha=0.3)
    plt.legend(title="Model Family", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

for target, fam_dict in baseline_family_auc_tracker.items():
    plt.figure(figsize=(10, 6))
    for fam, auc_list in fam_dict.items():
        plt.plot(range(1, num_runs + 1), auc_list, marker='o', label=fam)
    plt.title(f"Learning Curve (Baseline AUC across runs) — {target}")
    plt.xlabel("Run")
    plt.ylabel("AUC")
    plt.xticks(range(1, num_runs + 1))
    plt.grid(True, alpha=0.3)
    plt.legend(title="Baseline Family", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
# ------------------ SHUTDOWN H2O ------------------
h2o.shutdown(prompt=False)

